In [1]:
import pandas as pd
import os
from pinecone import Pinecone, ServerlessSpec
from dotenv import load_dotenv, find_dotenv
import pinecone
from sentence_transformers import SentenceTransformer

In [2]:
files = pd.read_csv("course_descriptions.csv", encoding = "ANSI")

In [3]:
def create_course_description(row):
    return f'''The course name is {row["course_name"]}, the slug is {row["course_slug"]},
            the technology is {row["course_technology"]} and the course topic is {row["course_topic"]}'''

In [4]:
pd.set_option('display.max_rows', 106)
files['course_description_new'] = files.apply(create_course_description, axis = 1)
print(files["course_description_new"])

0      The course name is Introduction to Tableau, th...
1      The course name is The Complete Data Visualiza...
2      The course name is Introduction to R Programmi...
3      The course name is Data Preprocessing with Num...
4      The course name is Introduction to Data and Da...
5      The course name is Data Cleaning and Preproces...
6      The course name is Introduction to Business An...
7      The course name is Data Analysis with Excel Pi...
8      The course name is SQL, the slug is sql,\n    ...
9      The course name is Credit Risk Modeling in Pyt...
10     The course name is Python Programmer Bootcamp,...
11     The course name is SQL + Tableau + Python, the...
12     The course name is Introduction to Jupyter, th...
13     The course name is Statistics, the slug is sta...
14     The course name is Mathematics, the slug is ma...
15     The course name is Introduction to Excel, the ...
16     The course name is Probability, the slug is pr...
17     The course name is Start

In [5]:
%load_ext dotenv
%dotenv

In [6]:
load_dotenv(find_dotenv(), override = True)

True

In [7]:
pc = Pinecone(
    api_key=os.environ.get("PINECONE_API_KEY")
)

In [8]:
model = SentenceTransformer('multi-qa-distilbert-cos-v1')

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

In [9]:
index_name = "my-index"
dimension = model.get_embedding_dimension()
metric = "cosine"

In [10]:
if index_name in [index.name for index in pc.list_indexes()]:
    pc.delete_index(index_name)
    print(f"{index_name} succesfully deleted.")
else:
     print(f"{index_name} not in index list.")

my-index succesfully deleted.


In [11]:
pc.create_index(
    name = index_name, 
    dimension = dimension, 
    metric = metric, 
    spec = ServerlessSpec(
        cloud = "aws", 
        region = "us-east-1")
    )

IndexModel(name='my-index', metric='cosine', status=IndexStatus(ready=True, state='Ready'), spec=IndexSpec(serverless=ServerlessSpecInfo(cloud='aws', region='us-east-1', read_capacity={'mode': 'OnDemand', 'status': {'state': 'Ready', 'current_shards': None, 'current_replicas': None}}, source_collection=None, schema=None), pod=None, byoc=None), host='https://my-index-4l0pdu7.svc.aped-4627-b74a.pinecone.io', private_host=None, vector_type='dense', dimension=768, deletion_protection='disabled', tags=None, embed=None, created_at=None)

In [12]:
index = pc.Index(index_name)

In [13]:
def create_embeddings(row):
    return model.encode(
        row["course_description_new"],
        show_progress_bar=False
    )

In [14]:
files["embedding"] = files.apply(create_embeddings, axis = 1)

In [15]:
vectors_to_upsert = [(str(row["course_name"]), row["embedding"].tolist()) for _, row in files.iterrows()]
index.upsert(vectors = vectors_to_upsert)

print("Data upserted to Pinecone index")

Data upserted to Pinecone index


In [16]:
query = "clustering"
query_embedding = model.encode(query, show_progress_bar = False).tolist()

In [17]:
query_results = index.query(
    vector = [query_embedding],
    top_k = 12, 
    include_values = True
)

In [18]:
query_results

QueryResponse(matches=[ScoredVector(id='Machine Learning with K-Nearest Neighbors', score=0.247339249, values=[-0.010773737, 0.0181405265, 0.0449687056, ...765 more]), ScoredVector(id='Machine Learning with Decision Trees and Random Forests', score=0.113718994, values=[0.010465987, 0.0480517857, -0.00813466311, ...765 more]), ScoredVector(id='Growth Analysis with SQL, Python, and Tableau  ', score=0.18820478, values=[0.0235343613, 0.033005055, 0.0305121578, ...765 more]), ScoredVector(id='Mastering Key Performance Indicators (KPIs)', score=0.146440521, values=[-0.00722010853, 0.0632523596, 0.0342653878, ...765 more]), ScoredVector(id='Customer Churn Analysis with SQL and Tableau', score=0.118093498, values=[0.0196716059, 0.0597109757, -0.00598678086, ...765 more]), ScoredVector(id='Probability', score=0.132458687, values=[0.029059086, 0.0275687892, 0.0196259413, ...765 more]), ScoredVector(id='Machine Learning with Ridge and Lasso Regression', score=0.128593445, values=[-0.0135004446, 

In [24]:
score_threshold = 0.1
for match in query_results["matches"]:
    if match['score'] >= score_threshold:
        print(f"Matched item ID: {match['id']}, score: {match['score']}")

Matched item ID: Machine Learning with K-Nearest Neighbors, score: 0.247339249
Matched item ID: Machine Learning with Decision Trees and Random Forests, score: 0.113718994
Matched item ID: Growth Analysis with SQL, Python, and Tableau  , score: 0.18820478
Matched item ID: Mastering Key Performance Indicators (KPIs), score: 0.146440521
Matched item ID: Customer Churn Analysis with SQL and Tableau, score: 0.118093498
Matched item ID: Probability, score: 0.132458687
Matched item ID: Machine Learning with Ridge and Lasso Regression, score: 0.128593445
Matched item ID: Statistics, score: 0.112636566
Matched item ID: Data-Driven Business Growth, score: 0.123642929
Matched item ID: Product Management for AI & Data Science, score: 0.113718033
Matched item ID: Data Analysis with Excel Pivot Tables, score: 0.115783699
Matched item ID: Communication and Presentation Skills for Analysts and Managers, score: 0.108019829
